In [6]:
import sys
print(sys.executable)

/opt/homebrew/Caskroom/miniforge/base/envs/mgh_analytics/bin/python


In [7]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

#1 Setting up Parameters - using seed to generate pseudo random number
np.random.seed(42)
days = 365
start_date = datetime(2025, 1, 1)
unit_name = "Bigelow-7-Surgical"

#2 Create a Date-Range for one full year
date_list = [start_date + timedelta(days=x) for x in range(days)]

#3 Create Synthetic Metrics
data = {
    'Date': date_list,
    'Unit': unit_name,
    #Patient Census: How many patients are hospitalized (avg 25, max 30)
    'Patient_Census': np.random.randint(20, 31, size=days),
    #Budgeted HPPD (Hours per Patient Day): MGH standard is 8.0
    'Target_HPPD': 8.0,
}

df = pd.DataFrame(data)

#4 Calculate Required vs Actual
#Budgeted_Hours = How many hours we SHOULD have worked based on patient count?
df['Budgeted_Hours'] = df['Patient_Census'] * df['Target_HPPD']

#Actual_Hours: Adding "noise" to simulate real life (sick calls, overtime, etc.)
#This creates a variance between -15% and +15%
df['Actual_Hours'] = df['Budgeted_Hours'] * np.random.uniform(0.85, 1.15, size=days)

#5 Calculate Key Performance Indicators (KPIs)
df['Variance_hours'] = df['Actual_Hours'] - df['Budgeted_Hours']
df['Actual_HPPD'] = df['Actual_Hours'] / df['Patient_Census']

#Rounding for clean data
df = df.round(2)

#6 Export to CSV (source file)
df.to_csv('mgh_staffing_data.csv', index=False)

print("Success! 'mgh_staffing_data.csv' has been created in your folder.")
print(df.head())

Success! 'mgh_staffing_data.csv' has been created in your folder.
        Date                Unit  Patient_Census  Target_HPPD  Budgeted_Hours  \
0 2025-01-01  Bigelow-7-Surgical              26          8.0           208.0   
1 2025-01-02  Bigelow-7-Surgical              23          8.0           184.0   
2 2025-01-03  Bigelow-7-Surgical              30          8.0           240.0   
3 2025-01-04  Bigelow-7-Surgical              27          8.0           216.0   
4 2025-01-05  Bigelow-7-Surgical              24          8.0           192.0   

   Actual_Hours  Variance_hours  Actual_HPPD  
0        192.79          -15.21         7.41  
1        190.16            6.16         8.27  
2        209.87          -30.13         7.00  
3        183.94          -32.06         6.81  
4        199.37            7.37         8.31  


/var/folders/pq/3wzt26_92n93nn_44n30bynh0000gq/T/ipykernel_95391/215158555.py:39: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  df = df.round(2)


In [ ]:
import pandas as pd
import urllib
from sqlalchemy import create_engine

# 1. Load the CSV
# Make sure the file name matches exactly what you saved earlier
df = pd.read_csv('mgh_staffing_data.csv')

# 2. Database Credentials
# Use the password you set in the Docker command (Ammu12345!)
user = "sa"
password = "Ammu12345!" # <--- Update this to your actual password
host = "localhost"
port = "1433"
database = "master"

# 3. Create the Connection String for Mac/ARM
# We use 'TrustServerCertificate=yes' because local Docker containers 
# don't usually have SSL certificates set up.
params = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 18 for SQL Server}};"
    f"SERVER={host},{port};"
    f"DATABASE={database};"
    f"UID={user};"
    f"PWD={password};"
    "Encrypt=yes;"
    "TrustServerCertificate=yes;"
)

# 4. Create Engine and Upload
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

try:
    df.to_sql('StaffingData', con=engine, if_exists='replace', index=False)
    print("🚀 Success! Data is now in the 'StaffingData' table in SQL Server.")
except Exception as e:
    print(f"❌ Error: {e}")

🚀 Success! Data is now in the 'StaffingData' table in SQL Server.


In [1]:
import pyodbc
print(pyodbc.drivers())

['ODBC Driver 18 for SQL Server']


In [4]:
import pandas as pd
import urllib.parse
import plotly.graph_objects as go
from sqlalchemy import create_engine

# 1. Database Credentials 
# (Make sure these match your Docker setup!)
user = "sa"
password = "Ammu12345!" 
host = "localhost"
port = "1433"
database = "master"

# 2. Create the Connection String
params = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 18 for SQL Server}};"
    f"SERVER={host},{port};"
    f"DATABASE={database};"
    f"UID={user};"
    f"PWD={password};"
    "Encrypt=yes;"
    "TrustServerCertificate=yes;"
)

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# 3. Pull the Data
try:
    sql_query = "SELECT * FROM StaffingData ORDER BY Date"
    df_plot = pd.read_sql(sql_query, engine)
    print("✅ Data successfully pulled from SQL Server!")
    
    # 4. Create the Interactive Chart
    fig = go.Figure()

    # Budgeted Hours
    fig.add_trace(go.Scatter(
        x=df_plot['Date'], y=df_plot['Budgeted_Hours'],
        name='Budgeted Hours', line=dict(color='gray', dash='dash')
    ))

    # Actual Hours
    fig.add_trace(go.Scatter(
        x=df_plot['Date'], y=df_plot['Actual_Hours'],
        name='Actual Hours', line=dict(color='blue', width=2)
    ))

    fig.update_layout(
        title='MGH Nursing Unit: Staffing Volume & Budget Variance',
        xaxis_title='Date',
        yaxis_title='Nursing Hours',
        template='plotly_white',
        hovermode='x unified'
    )

    fig.show()

except Exception as e:
    print(f"❌ Error: {e}")

✅ Data successfully pulled from SQL Server!


In [ ]:
# Below is the MONTHLY SUMMARY Query
# Updated SQL Query using CONVERT instead of FORMAT to avoid CLR error
monthly_sql = """
SELECT 
    LEFT(CONVERT(VARCHAR, CAST([Date] AS DATE), 120), 7) AS Month,
    SUM(Patient_Census) AS Total_Census,
    SUM(Budgeted_Hours) AS Total_Budgeted,
    SUM(Actual_Hours) AS Total_Actual,
    SUM(Actual_Hours) - SUM(Budgeted_Hours) AS Monthly_Variance
FROM StaffingData
GROUP BY LEFT(CONVERT(VARCHAR, CAST([Date] AS DATE), 120), 7)
ORDER BY Month
"""

df_monthly = pd.read_sql(monthly_sql, engine)
print(df_monthly.head())

     Month  Total_Census  Total_Budgeted  Total_Actual  Monthly_Variance
0  2025-01           792          6336.0       6178.21           -157.79
1  2025-02           686          5488.0       5376.92           -111.08
2  2025-03           786          6288.0       6341.24             53.24
3  2025-04           749          5992.0       5992.12              0.12
4  2025-05           783          6264.0       6228.06            -35.94


In [ ]:
import plotly.express as px

# 1. Create a 'Color' column logic
# In hospital ops, Positive Variance = Over Budget (Red)
# Negative Variance = Under Budget (Green)
df_monthly['Color_Label'] = df_monthly['Monthly_Variance'].apply(
    lambda x: 'Over Budget' if x > 0 else 'Under/On Budget'
)

# 2. Create the Bar Chart
fig_monthly = px.bar(
    df_monthly, 
    x='Month', 
    y='Monthly_Variance',
    color='Color_Label',
    # MGH/Professional Color Palette
    color_discrete_map={
        'Over Budget': '#d62728',      # Muted Red
        'Under/On Budget': '#2ca02c'   # Success Green
    },
    title='MGH Nursing Unit: Monthly Labor Variance (Hours)',
    labels={
        'Monthly_Variance': 'Variance (Hours)', 
        'Month': 'Fiscal Month',
        'Color_Label': 'Budget Status'
    },
    hover_data=['Total_Census', 'Total_Actual'] # Shows extra info when you hover
)

# 3. Clean up the design for an "Executive" look
fig_monthly.update_layout(
    template='plotly_white',
    xaxis_tickangle=-45,
    showlegend=True,
    # Adds a horizontal line at 0 for easy reference
    shapes=[dict(
        type='line', yref='y', y0=0, y1=0, xref='paper', x0=0, x1=1,
        line=dict(color="black", width=1)
    )]
)

fig_monthly.show()